[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/l/lab-l1-artifact-descent.ipynb)

# LAB·L1 · One program, every artifact

**Hardware:** any machine. Everything below runs on CPU, in one process, in about an hour.

One function: sixteen rows wide, five matmuls and a softmax, the shape of an attention block with the batch dimension taken away. By the end of this notebook you will have read that same function written down five times, as Python, as a jaxpr, as StableHLO, as scheduled HLO with fusions in it, and as the LLVM IR the CPU backend hands to LLVM. None of it is a diagram of a compiler. Every artifact below is printed by the machine you are sitting at, on the run you are about to do.

The rule for this lab, the same one the rest of the path uses: predict before you run. Each level opens with two or three questions about the dump you have not seen yet. Write your answers in the prediction cell first, in your own words. Reading a dump you already guessed at is a different act from reading one cold, and only the first one builds the skill.

Two branches of the stack are missing here, deliberately. Which ones, and why they cannot run on this machine, is the second-to-last section.

In [ ]:
import inspect
import os
import re
import shutil

# The dump flag has to be in the environment before jax picks a backend, so
# this cell owns all of the setup: flags first, then imports, then the program.
DUMP = "/tmp/xla-descent"
shutil.rmtree(DUMP, ignore_errors=True)
os.environ["XLA_FLAGS"] = f"--xla_dump_to={DUMP}"

import jax
import jax.numpy as jnp

print(jax.__version__, jax.devices())


def block(x, wq, wk, wv):
    q, k, v = x @ wq, x @ wk, x @ wv
    s = (q @ k.T) / jnp.sqrt(jnp.float32(q.shape[-1]))
    return jax.nn.softmax(s, axis=-1) @ v


D = 32
x = jnp.ones((16, D), jnp.float32)
w = jnp.ones((D, D), jnp.float32)


def strip_metadata(text):
    """HLO tags every instruction with the file and line it came from. Those
    paths describe the machine that ran the compile rather than the program,
    so they come off before anything gets printed."""
    return re.sub(r", metadata=\{[^}]*\}", "", text)


print("program ready:", x.shape, "against three weights of", w.shape)

## Level 1 · the source

Python is where the program gets decided, and the last place where it is only decided. The three projections run in the order you wrote them because you wrote them in that order. Nothing here knows how wide `x` is.

Before you run the next cell:

- one line of `block` turns into more than one machine operation. Which line, and roughly how many?
- what does this artifact know about the shape of `x`?
- `q @ k.T` asks for a transpose. Does anything at this level promise a transpose actually happens?

**your prediction:**

In [ ]:
print(inspect.getsource(block))
print("attribute names Python will go looking for:", block.__code__.co_names)
print("locals it reserved slots for:              ", block.__code__.co_varnames)
print("constants baked into the code object:      ", block.__code__.co_consts)

Python's own artifact is the code object, and what it carries is names. `co_varnames` lists the eight locals, `co_names` lists the attributes it will chase at call time, `T` and `softmax` among them. Shapes are absent, dtypes are absent, the device is absent, because none of them exist yet. Call it on a 4096 by 512 array instead and the code object is byte for byte the same.

That absence is what makes the next four artifacts possible. Every level below this one gets to assume the shapes are known, because the trace that produces them starts by asking for them.

## Level 2 · the jaxpr

`jax.make_jaxpr` calls the function on tracers, values that carry a shape and a dtype and no numbers, and writes down every primitive that gets called along the way. Nothing compiles. This is the recording.

- `jax.nn.softmax` is one call in the source. How many equations do you expect it to leave behind?
- the source binds four intermediates, `q`, `k`, `v`, `s`. How many names do you expect the jaxpr to bind?
- pick any name in the jaxpr you are about to read. How many times can it show up on the left of an `=`?

**your prediction:**

In [ ]:
print(jax.make_jaxpr(block)(x, w, w, w))

Three lines of arithmetic, seventeen equations, and softmax alone accounts for nine of them: a max down the row, a max against negative infinity, a broadcast, a `stop_gradient`, a subtract, an `exp`, a sum, a second broadcast, a divide. Written out like that the numerical care is visible in the artifact. Subtracting the row max before exponentiating is what stops the exponent from overflowing, and you did not write it. `jax.nn.softmax` did, once, on your behalf.

Now read only the left-hand sides. Every equation binds a fresh letter and no letter is ever reassigned. The property has a name, static single assignment, and the next cell checks it rather than trusting the eye.

In [ ]:
jaxpr = jax.make_jaxpr(block)(x, w, w, w).jaxpr
bound = [str(v) for eqn in jaxpr.eqns for v in eqn.outvars]

print(f"{len(jaxpr.eqns)} equations")
print(f"{len(bound)} names bound, {len(set(bound))} of them distinct")
print("every name defined exactly once:", len(bound) == len(set(bound)))
print()
print("primitives, in order:")
print(" ", [str(eqn.primitive) for eqn in jaxpr.eqns])

Seventeen bindings, seventeen distinct names. To find where a value came from you read one line, the one that binds it, and there is never a second definition to reconcile against it. A pass that wants to move an instruction has one question to answer, whether the inputs are computed yet, and the artifact answers that by construction rather than by analysis. The lesson at kernels.rudrite.com/l/jaxpr/one-name-one-definition argues the property in prose. The line above is the same claim, checked on your run.

## Level 3 · StableHLO

`.lower()` hands the jaxpr to JAX's lowering rules and gets MLIR back. No optimization has happened. This is the program as it leaves JAX, before XLA has formed an opinion about it.

- two dialects appear in the dump. Which operations belong to which?
- the jaxpr carried a `transpose`. Is it still here?
- jaxpr equations were named `a`, `b`, `c`. What are values called now, and what does the naming tell you about who wrote the text?

**your prediction:**

In [ ]:
lowered = jax.jit(block).lower(x, w, w, w)
print(lowered.as_text())

Two prefixes cover the whole dump. `func.func` and its `return` come from the func dialect, the part of MLIR that knows what a function is and nothing whatever about tensors. Everything doing arithmetic is prefixed `stablehlo`. One module holding operations from several dialects at once is the ordinary case in MLIR, and it is the reason the level is called a dialect rather than a language.

Values are `%0`, `%1`, `%2`, numbered in the order they were produced. Same single-assignment property as the jaxpr, different notation, and the numbering makes it obvious a machine wrote this. Types ride along on every value. `tensor<16x32xf32>` states rank, shape and element type on the operation itself, so nothing downstream has to infer them.

The transpose is still here, at `%3`, and the five `dot_general` operations are still five separate operations. Nobody has decided yet that any of this can be avoided.

## Level 4 · the optimized HLO

`.compile()` runs the backend pipeline, a few hundred passes deep, and what comes out is scheduled, laid out and fused. It is still HLO and still the same program, and it no longer looks much like the thing that went in.

- the transpose survived level 3. Does it survive the pipeline?
- `jnp.sqrt(jnp.float32(32))` is a square root of a constant. Where does it end up?
- some instructions in the dump call into a separate computation instead of doing arithmetic themselves. Which ones, and what is inside them?

**your prediction:**

In [ ]:
compiled = lowered.compile()
hlo = strip_metadata(compiled.as_text())
print(hlo)

The transpose is gone. `%dot` produces `f32[32,16]` straight out of the weight and the input with `lhs_contracting_dims={0}`, which is the same arithmetic read down the other axis. A dot contracts along whichever dimension it is told to, so the cheapest way to serve a transposed operand is to move no data at all.

The square root is gone too, folded to the literal `0.176776692`, and it turns up twice, once inside each fusion that needs it. Duplicating a scalar constant into two fusions costs nothing and saves a buffer.

Three instructions in `ENTRY main` end in `_fusion` and carry `calls=%fused_computation...`. Open `fused_computation.1`, the body behind `subtract_exponential_fusion`, and it holds a multiply, three broadcasts, a maximum, a subtract and an exponential in one unit. Those were separate passes over a 16 by 16 array in the jaxpr, each one reading and writing memory. They are a single pass now, and the values between them never get a buffer of their own. Fusion changed nothing about the arithmetic. It changed how many times the data gets read and written.

The five `dot` instructions were left alone, unfused. The next level says why.

## Level 5 · the LLVM IR

The dump directory has been filling up this whole time. XLA's CPU backend turns each computation it means to generate code for into LLVM IR, then hands that to LLVM, which is where vectorization and register allocation happen. `--xla_dump_to` keeps both sides of the handoff, `ir-no-opt` before LLVM's own pipeline and `ir-with-opt` after it.

- ENTRY held five dots, three fusions and a reduce. How many LLVM functions do you expect?
- LLVM IR is single assignment too, and it has loops. What does a loop counter look like when a name can only be assigned once?
- this is x86 after LLVM's optimizer. What width do you expect the float arithmetic to be?

**your prediction:**

In [ ]:
artifacts = sorted(f for f in os.listdir(DUMP) if "jit_block" in f)
print(f"{len(artifacts)} files on disk for this one compile:")
for f in artifacts:
    print("  ", f)

parts = [f for f in artifacts if "ir-with-opt" in f]
llvm_fns = []
for p in parts:
    with open(os.path.join(DUMP, p)) as fh:
        llvm_fns += [ln.split("@")[1].split("(")[0] for ln in fh if ln.startswith("define")]

print()
print("LLVM functions XLA emitted:", llvm_fns)

In [ ]:
with open(os.path.join(DUMP, parts[0])) as fh:
    print(fh.read())

Four functions, and every name is one you already read in the HLO: the three fusions and the reduce. Nothing new was invented at this level, and nothing was merged.

Inside the first one, `%multiply_reduce_fusion.invar_address.dim.0.05 = phi i64 [ 0, %1 ], [ %invar.inc, %middle.block ]` is the loop counter. Single assignment forbids incrementing a name in place, so LLVM writes the merge down as an instruction of its own. A phi takes one value if control arrived from the entry block and another if it arrived from the loop body. Every SSA form with loops needs some version of this, and one phi read carefully is enough to recognize all the others.

`<8 x float>` sits on nearly every arithmetic line, and `llvm.vector.reduce.fmaximum.v8f32` does the row max eight lanes at a time. No HLO instruction asked for that. LLVM's vectorizer decided it, for this target, at this optimization level, after XLA was finished. The vector width you see depends on the CPU this ran on.

In [ ]:
fusions = sorted(set(re.findall(r"^\s*%?([\w.]+_fusion) = ", hlo, re.M)))
dots = re.findall(r"^\s*(?:ROOT\s+)?%?(dot[\w.]*) = ", hlo, re.M)

print("fusion names in the HLO:      ", fusions)
print("the same names in the LLVM IR:", sorted(set(fusions) & set(llvm_fns)))
print()
print(f"dot instructions in the HLO: {len(dots)} → {dots}")
print("LLVM functions with 'dot' in the name:", [f for f in llvm_fns if "dot" in f])

Five dots in the HLO, none of them in the LLVM IR. The CPU backend does not generate code for a matmul at all. It emits a call into the XLA runtime, which dispatches to a library kernel someone tuned by hand, and the LLVM modules only cover the work around that call. Every backend on every target splits this way. Some operations get compiled, some get called out to, and which is which is a backend decision you can read off a dump instead of speculating about.

It matters that the names survive that crossing. `subtract_exponential_fusion` is a fusion decision made in HLO and a symbol in the object file after lowering, so a profile that blames that symbol points back at a specific group of jaxpr equations and one line of your Python.

## The levels this machine cannot show you

Five artifacts, one process, no hardware past the CPU under it. The stack does not stop at level 5, and the honest thing is to name what is missing rather than print something shaped like it.

In [ ]:
for backend in ("cpu", "gpu", "tpu"):
    try:
        print(f"{backend}: {jax.device_count(backend=backend)} device(s)")
    except RuntimeError as e:
        print(f"{backend}: absent · {str(e).splitlines()[0][:70]}")

On a CUDA machine the descent keeps going past level 5 and forks. XLA:GPU routes some fusions through Triton, which has two dialects of its own, TTIR while the work is still per-tensor and TTGIR once it has been assigned to warps, and both lower into LLVM IR again in its NVPTX flavour. LLVM emits PTX, a virtual ISA that ships as text and stays forward compatible across generations. `ptxas` compiles PTX into SASS, the real instruction set of one GPU generation, and `nvdisasm` is what prints SASS back out. On a TPU the fork goes the other way: Mosaic is the public level, and the LLO underneath it is not documented outside Google.

None of that runs here, so none of it is printed here. The lessons that teach those levels quote dumps captured on machines that had the hardware, each one cited where it appears:

- TTIR through PTX on one Triton kernel: kernels.rudrite.com/xla/codegen/triton-end-to-end
- the ISA contract, PTX against SASS: kernels.rudrite.com/s/machine/the-isa-contract
- VLIW bundles and the LLO edge: kernels.rudrite.com/l/tpu/vliw-bundles-and-llo

If you do have a GPU, the exercise that finishes this lab is to run the same `block` there and diff the level-4 dump against the one you just read. The fusion decisions come out different, and the reason is the scheduling hardware rather than a different compiler.

## Mark it run

Five printings of one function: source, jaxpr, StableHLO, optimized HLO, LLVM IR. You checked single assignment instead of believing it, watched a transpose stop existing, found the operations that became one fusion, and matched a fusion name across the boundary between two IRs.

Handed an unlabeled dump now, you have a procedure. Look at what values are named, look at whether types ride on the operations, look at whether tensors are still tensors or have become pointers and loops. Those three answers place any artifact on this ladder before you have read a single instruction.

Go back to the chapter page and tick LAB·L1 as run.